# GDS(Graph Data Science)를 활용한 영화 추천 시스템

이 노트북에서는 Neo4j의 GDS 라이브러리를 활용하여 영화 추천 시스템을 구축합니다.



## 학습 목표
1. GDS 라이브러리의 기본 사용법 이해
2. 그래프 프로젝션(Graph Projection) 방법 학습
3. PageRank 알고리즘으로 인기 영화 찾기
4. Node Similarity로 유사 영화 추천
5. Community Detection으로 영화 그룹 분석
6. 실제 추천 시스템 구현


## 1. GDS란?

**GDS(Graph Data Science)**는 Neo4j에서 제공하는 그래프 데이터 과학 라이브러리입니다.

### 주요 특징
- **그래프 알고리즘**: PageRank, Centrality, Community Detection 등
- **그래프 임베딩**: 노드를 벡터로 변환하여 ML 모델에 활용
- **성능 최적화**: 대규모 그래프 데이터를 효율적으로 처리
- **Python 통합**: `graphdatascience` 라이브러리를 통해 쉽게 사용

### 추천 시스템에서의 활용
- **PageRank**: 인기 있는 영화/사용자 찾기
- **Node Similarity**: 유사한 영화/사용자 찾기
- **Community Detection**: 비슷한 취향의 사용자 그룹 찾기


## 2. 필요한 라이브러리 설치 및 import


In [ ]:
import pandas as pd
import ast
from neo4j import GraphDatabase
from graphdatascience import GraphDataScience

# 경고 메시지 숨기기 (선택사항)
import warnings
warnings.filterwarnings('ignore')


## 3. Neo4j 연결 설정


In [ ]:
# ============================================
# Neo4j 연결 정보 설정
# ============================================
URI = "bolt://localhost:7687"
USERNAME = "neo4j"
PASSWORD = "test1234"

In [ ]:
# ============================================
# Neo4j 드라이버 생성
# ============================================
driver = GraphDatabase.driver(URI, auth=(USERNAME, PASSWORD))

In [ ]:
# ============================================
# GDS 객체 생성
# ============================================
# GraphDataScience: GDS 라이브러리의 메인 클래스
# driver: Neo4j 드라이버 객체를 전달
gds = GraphDataScience(URI, auth=(USERNAME, PASSWORD))

In [ ]:
# 연결 테스트
try:
    result = driver.session().run("RETURN 1 as test")
    print("✓ Neo4j 연결 성공!")
except Exception as e:
    print(f"✗ Neo4j 연결 실패: {e}")


## 4. 데이터 로딩 및 전처리

영화 데이터셋을 로드하고 Neo4j에 적합한 형태로 전처리합니다.


In [ ]:
# 영화 메타데이터 로드
movies_df = pd.read_csv('data/movies_metadata.csv', low_memory=False)
print(f"영화 데이터: {len(movies_df)}개")


In [ ]:
# 평점 데이터 로드
ratings_df = pd.read_csv('data/ratings_small.csv')
print(f"평점 데이터: {len(ratings_df)}개")


In [ ]:
# 데이터 샘플링 (실습 속도를 위해)
# 전체 데이터를 사용하려면 이 부분을 주석 처리하세요
SAMPLE_SIZE = 5000  # 영화 수 제한
if len(movies_df) > SAMPLE_SIZE:
    movies_df = movies_df.head(SAMPLE_SIZE)
    movie_ids = set(movies_df['id'].astype(str))
    ratings_df = ratings_df[ratings_df['movieId'].astype(str).isin(movie_ids)]
    print(f"샘플링 후 - 영화: {len(movies_df)}개, 평점: {len(ratings_df)}개")

print("\n데이터 로딩 완료!")

In [ ]:
# ============================================
# 데이터 전처리: 장르 파싱
# ============================================
def parse_genres(genres_str):
    """
    장르 문자열을 파싱하여 장르 이름 리스트 반환
    
    예: "[{'id': 16, 'name': 'Animation'}, ...]" 
    -> ['Animation', 'Comedy', ...]
    """
    if pd.isna(genres_str) or genres_str == '':
        return []
    
    try:
        # 문자열을 딕셔너리 리스트로 변환
        genres_list = ast.literal_eval(genres_str)
        # 장르 이름만 추출
        return [g['name'] for g in genres_list if isinstance(g, dict) and 'name' in g]
    except:
        return []
        

In [ ]:
# 영화 데이터에 장르 리스트 추가
movies_df['genres_list'] = movies_df['genres'].apply(parse_genres)
print("장르 파싱 완료!")


In [ ]:
# 샘플 확인
print("\n샘플 영화 데이터:")
sample_movie = movies_df[['title', 'genres_list']].head(3)
for idx, row in sample_movie.iterrows():
    print(f"  - {row['title']}: {row['genres_list']}")


## 5. Neo4j 그래프 생성

영화, 사용자, 장르를 노드로, 평점과 장르 관계를 엣지로 생성합니다.


In [ ]:
# ============================================
# 기존 데이터 삭제 (선택사항)
# ============================================
# 주의: 모든 데이터가 삭제됩니다!
def clear_database():
    """Neo4j의 모든 노드와 관계를 삭제"""
    with driver.session() as session:
        session.run("MATCH (n) DETACH DELETE n")
    print("기존 데이터 삭제 완료!")
    

In [ ]:
clear_database()


In [ ]:
# ============================================
# 영화 노드 생성
# ============================================
def create_movies(session, movies_df):
    """영화 노드를 Neo4j에 생성"""
    query = """
    UNWIND $movies AS movie
    MERGE (m:Movie {movieId: movie.id})
    SET m.title = movie.title,
        m.overview = movie.overview,
        m.voteAverage = movie.vote_average,
        m.voteCount = movie.vote_count,
        m.releaseDate = movie.release_date
    """
    
    movies = []
    for _, row in movies_df.iterrows():
        movies.append({
            'id': str(row['id']),
            'title': str(row['title']) if pd.notna(row['title']) else '',
            'overview': str(row['overview']) if pd.notna(row['overview']) else '',
            'vote_average': float(row['vote_average']) if pd.notna(row['vote_average']) else 0.0,
            'vote_count': int(row['vote_count']) if pd.notna(row['vote_count']) else 0,
            'release_date': str(row['release_date']) if pd.notna(row['release_date']) else ''
        })
    
    session.run(query, movies=movies)
    print(f"영화 노드 {len(movies)}개 생성 완료!")
    

In [ ]:
with driver.session() as session:
    create_movies(session, movies_df)


In [ ]:
# ============================================
# 장르 노드 및 관계 생성
# ============================================
def create_genres(session, movies_df):
    """장르 노드와 영화-장르 관계 생성"""
    query = """
    UNWIND $movie_genres AS mg
    MATCH (m:Movie {movieId: mg.movieId})
    MERGE (g:Genre {name: mg.genre})
    MERGE (m)-[:HAS_GENRE]->(g)
    """
    
    movie_genres = []
    for _, row in movies_df.iterrows():
        movie_id = str(row['id'])
        genres = row['genres_list']
        for genre in genres:
            movie_genres.append({
                'movieId': movie_id,
                'genre': genre
            })
    
    if movie_genres:
        session.run(query, movie_genres=movie_genres)
        print(f"장르 관계 {len(movie_genres)}개 생성 완료!")


In [ ]:
with driver.session() as session:
    create_genres(session, movies_df)


In [ ]:
# ============================================
# 사용자 노드 및 평점 관계 생성
# ============================================
def create_ratings(session, ratings_df):
    """사용자 노드와 평점 관계 생성"""
    # 배치 처리로 성능 향상
    batch_size = 1000
    total_ratings = len(ratings_df)
    
    query = """
    UNWIND $ratings AS rating
    MERGE (u:User {userId: rating.userId})
    MERGE (m:Movie {movieId: rating.movieId})
    MERGE (u)-[r:RATED]->(m)
    SET r.rating = rating.rating,
        r.timestamp = rating.timestamp
    """
    
    processed = 0
    for i in range(0, total_ratings, batch_size):
        batch = ratings_df.iloc[i:i+batch_size]
        ratings = []
        for _, row in batch.iterrows():
            ratings.append({
                'userId': str(int(row['userId'])),
                'movieId': str(int(row['movieId'])),
                'rating': float(row['rating']),
                'timestamp': int(row['timestamp'])
            })
        
        session.run(query, ratings=ratings)
        processed += len(ratings)
        if (i // batch_size + 1) % 10 == 0:
            print(f"  진행 중... {processed}/{total_ratings}")
    
    print(f"평점 관계 {processed}개 생성 완료!")


In [ ]:
with driver.session() as session:
    create_ratings(session, ratings_df)


In [ ]:
# ============================================
# 그래프 통계 확인
# ============================================
def get_graph_stats(session):
    """그래프의 노드와 관계 개수 확인"""
    query = """
    MATCH (n)
    RETURN labels(n)[0] AS label, count(n) AS count
    ORDER BY label
    """
    result = session.run(query)
    
    print("=== 노드 통계 ===")
    for record in result:
        print(f"  {record['label']}: {record['count']}개")
    
    query = """
    MATCH ()-[r]->()
    RETURN type(r) AS type, count(r) AS count
    ORDER BY type
    """
    result = session.run(query)
    
    print("\n=== 관계 통계 ===")
    for record in result:
        print(f"  {record['type']}: {record['count']}개")


In [ ]:
with driver.session() as session:
    get_graph_stats(session)


## 6. GDS 그래프 프로젝션

GDS 알고리즘을 실행하기 전에 Neo4j 데이터를 GDS 형식으로 프로젝션해야 합니다.

### 그래프 프로젝션이란?
- Neo4j의 실제 데이터를 메모리 내 그래프 구조로 변환
- 알고리즘 실행 속도 향상
- 필요한 노드와 관계만 선택하여 프로젝션 가능


In [ ]:
# ============================================
# 영화-사용자 평점 그래프 프로젝션
# ============================================
# 이 그래프는 사용자와 영화 간의 평점 관계를 나타냅니다.
# 추천 시스템의 핵심 그래프입니다.

G_movies_users, result = gds.graph.project(
    "movies-users-graph",  # 그래프 이름
    ["Movie", "User"],     # 노드 레이블
    "RATED",                # 관계 타입
    nodeProperties=["title", "voteAverage"],  # 노드 속성 (선택사항)
    relationshipProperties=["rating"]         # 관계 속성 (선택사항)
)

print(f"그래프 프로젝션 완료!")
print(f"  노드 수: {G_movies_users.node_count()}")
print(f"  관계 수: {G_movies_users.relationship_count()}")


In [ ]:
# ============================================
# 영화-장르 그래프 프로젝션
# ============================================
# 이 그래프는 영화와 장르 간의 관계를 나타냅니다.
# 유사 영화 찾기에 활용됩니다.

G_movies_genres, result = gds.graph.project(
    "movies-genres-graph",
    ["Movie", "Genre"],
    "HAS_GENRE"
)

print(f"그래프 프로젝션 완료!")
print(f"  노드 수: {G_movies_genres.node_count()}")
print(f"  관계 수: {G_movies_genres.relationship_count()}")


## 7. PageRank 알고리즘으로 인기 영화 찾기

**PageRank**는 노드의 중요도를 계산하는 알고리즘입니다. 
- 많은 사용자가 평점을 준 영화일수록 높은 점수
- 높은 평점을 받은 영화일수록 높은 점수
- 추천 시스템에서 "인기 영화"를 찾는 데 활용


In [ ]:
# ============================================
# PageRank 실행
# ============================================
# PageRank 알고리즘을 실행하여 각 영화의 중요도 점수를 계산합니다.

pagerank_result = gds.pageRank.stream(
    G_movies_users,           # 프로젝션된 그래프
    maxIterations=20,         # 최대 반복 횟수
    dampingFactor=0.85        # 댐핑 팩터 (일반적으로 0.85 사용)
)

# 결과를 DataFrame으로 변환
pagerank_df = pagerank_result.to_pandas()

print(f"PageRank 계산 완료!")
print(f"총 {len(pagerank_df)}개 노드의 점수 계산됨")
print("\n상위 10개 인기 영화:")
print(pagerank_df.head(10))


In [ ]:
# ============================================
# PageRank 결과를 Neo4j에 저장
# ============================================
# 계산된 PageRank 점수를 영화 노드의 속성으로 저장합니다.
# 이렇게 하면 나중에 Cypher 쿼리로 쉽게 조회할 수 있습니다.

gds.pageRank.write(
    G_movies_users,
    writeProperty="pagerank",  # 저장할 속성 이름
    maxIterations=20,
    dampingFactor=0.85
)

print("PageRank 점수가 영화 노드에 저장되었습니다!")

# 저장된 결과 확인
with driver.session() as session:
    query = """
    MATCH (m:Movie)
    WHERE m.pagerank IS NOT NULL
    RETURN m.title AS 영화, m.pagerank AS PageRank점수
    ORDER BY m.pagerank DESC
    LIMIT 10
    """
    result = session.run(query)
    
    print("\n=== PageRank 상위 10개 영화 ===")
    for i, record in enumerate(result, 1):
        print(f"{i:2d}. {record['영화']}: {record['PageRank점수']:.6f}")


## 8. Node Similarity로 유사 영화 찾기

**Node Similarity**는 두 노드가 얼마나 유사한지 계산하는 알고리즘입니다.
- 같은 사용자들이 평점을 준 영화는 유사하다고 판단
- 같은 장르를 가진 영화는 유사하다고 판단
- 추천 시스템에서 "이 영화와 비슷한 영화"를 찾는 데 활용


In [ ]:
# ============================================
# Node Similarity 실행 (사용자 평점 기반)
# ============================================
# 같은 사용자들이 평점을 준 영화들 간의 유사도를 계산합니다.

similarity_result = gds.nodeSimilarity.stream(
    G_movies_users,
    similarityMetric="JACCARD",  # 유사도 측정 방법: JACCARD 또는 COSINE
    topK=10,                      # 각 영화당 상위 10개 유사 영화만 저장
    similarityCutoff=0.1          # 최소 유사도 임계값 (0.1 미만은 제외)
)

similarity_df = similarity_result.to_pandas()
print(f"유사도 계산 완료!")
print(f"총 {len(similarity_df)}개의 유사 영화 쌍 발견")
print("\n샘플 유사 영화 쌍:")
print(similarity_df.head(10))


In [ ]:
# ============================================
# 특정 영화의 유사 영화 찾기
# ============================================
def find_similar_movies(movie_title, similarity_df, top_n=5):
    """
    특정 영화와 유사한 영화를 찾는 함수
    
    Args:
        movie_title: 찾고자 하는 영화 제목
        similarity_df: 유사도 결과 DataFrame
        top_n: 반환할 유사 영화 개수
    """
    # 영화 ID 찾기
    with driver.session() as session:
        query = """
        MATCH (m:Movie {title: $title})
        RETURN m.movieId AS movieId
        LIMIT 1
        """
        result = session.run(query, title=movie_title)
        record = result.single()
        
        if not record:
            print(f"영화 '{movie_title}'를 찾을 수 없습니다.")
            return
        
        movie_id = record['movieId']
    
    # 유사 영화 찾기
    similar = similarity_df[
        (similarity_df['node1'] == int(movie_id)) | 
        (similarity_df['node2'] == int(movie_id))
    ].copy()
    
    if len(similar) == 0:
        print(f"'{movie_title}'와 유사한 영화를 찾을 수 없습니다.")
        return
    
    # 유사도가 높은 순으로 정렬
    similar = similar.sort_values('similarity', ascending=False).head(top_n)
    
    # 영화 제목 가져오기
    with driver.session() as session:
        print(f"\n=== '{movie_title}'와 유사한 영화 (상위 {top_n}개) ===\n")
        for idx, row in similar.iterrows():
            # node1 또는 node2 중 movie_id가 아닌 쪽이 유사 영화
            other_id = row['node2'] if row['node1'] == int(movie_id) else row['node1']
            
            query = """
            MATCH (m:Movie {movieId: $movieId})
            RETURN m.title AS title, m.voteAverage AS rating
            """
            result = session.run(query, movieId=str(other_id))
            record = result.single()
            
            if record:
                print(f"  - {record['title']} (유사도: {row['similarity']:.4f}, 평점: {record['rating']})")

# 예제: "Toy Story"와 유사한 영화 찾기
find_similar_movies("Toy Story", similarity_df, top_n=5)


In [ ]:
# ============================================
# Node Similarity 결과를 Neo4j에 관계로 저장
# ============================================
# 유사도가 높은 영화 쌍을 SIMILAR 관계로 저장합니다.
# 이렇게 하면 나중에 Cypher 쿼리로 쉽게 조회할 수 있습니다.

gds.nodeSimilarity.write(
    G_movies_users,
    writeRelationshipType="SIMILAR",  # 저장할 관계 타입
    writeProperty="similarity",       # 유사도 점수를 관계 속성으로 저장
    similarityMetric="JACCARD",
    topK=10,
    similarityCutoff=0.1
)

print("유사 영화 관계가 저장되었습니다!")

# 저장된 결과 확인
with driver.session() as session:
    query = """
    MATCH (m1:Movie)-[s:SIMILAR]-(m2:Movie)
    WHERE m1.title = 'Toy Story'
    RETURN m2.title AS 유사영화, s.similarity AS 유사도
    ORDER BY s.similarity DESC
    LIMIT 5
    """
    result = session.run(query)
    
    print("\n=== Cypher로 조회한 유사 영화 ===")
    for record in result:
        print(f"  - {record['유사영화']}: {record['유사도']:.4f}")


## 9. Community Detection으로 영화 그룹 찾기

**Community Detection**은 그래프에서 밀집하게 연결된 노드 그룹을 찾는 알고리즘입니다.
- 비슷한 취향의 사용자 그룹 찾기
- 비슷한 특성을 가진 영화 그룹 찾기
- 추천 시스템에서 "이 그룹의 사용자들이 좋아하는 영화" 추천에 활용


In [ ]:
# ============================================
# Louvain 알고리즘으로 커뮤니티 탐지
# ============================================
# Louvain은 가장 널리 사용되는 커뮤니티 탐지 알고리즘입니다.
# 모듈성(modularity)을 최대화하여 그룹을 찾습니다.

community_result = gds.louvain.stream(
    G_movies_users,
    maxLevels=10,        # 최대 레벨 수
    maxIterations=10     # 각 레벨의 최대 반복 횟수
)

community_df = community_result.to_pandas()
print(f"커뮤니티 탐지 완료!")
print(f"총 {len(community_df)}개 노드가 {community_df['communityId'].nunique()}개 그룹으로 분류됨")
print("\n커뮤니티별 노드 개수:")
print(community_df['communityId'].value_counts().head(10))


In [ ]:
# ============================================
# 커뮤니티 결과를 Neo4j에 저장 및 분석
# ============================================
gds.louvain.write(
    G_movies_users,
    writeProperty="community"  # 저장할 속성 이름
)

print("커뮤니티 정보가 영화 노드에 저장되었습니다!")

# 각 커뮤니티의 대표 영화 확인
with driver.session() as session:
    query = """
    MATCH (m:Movie)
    WHERE m.community IS NOT NULL
    WITH m.community AS community, 
         collect(m.title) AS movies,
         avg(m.voteAverage) AS avgRating,
         count(m) AS movieCount
    WHERE movieCount >= 5
    RETURN community AS 커뮤니티ID, 
           movieCount AS 영화수,
           avgRating AS 평균평점,
           movies[0..5] AS 대표영화
    ORDER BY movieCount DESC
    LIMIT 10
    """
    result = session.run(query)
    
    print("\n=== 주요 커뮤니티 분석 ===")
    for record in result:
        print(f"\n커뮤니티 {record['커뮤니티ID']}:")
        print(f"  영화 수: {record['영화수']}개")
        print(f"  평균 평점: {record['평균평점']:.2f}")
        print(f"  대표 영화: {', '.join(record['대표영화'])}")


## 10. 추천 시스템 구현

위에서 학습한 GDS 알고리즘들을 활용하여 실제 추천 시스템을 구현합니다.

### 추천 전략
1. **인기 영화 추천**: PageRank 점수가 높은 영화
2. **유사 영화 추천**: 사용자가 본 영화와 유사한 영화
3. **커뮤니티 기반 추천**: 같은 커뮤니티의 다른 사용자들이 좋아한 영화


In [ ]:
# ============================================
# 추천 시스템 클래스 구현
# ============================================
class MovieRecommender:
    """GDS 알고리즘을 활용한 영화 추천 시스템"""
    
    def __init__(self, driver, gds):
        self.driver = driver
        self.gds = gds
    
    def recommend_by_popularity(self, top_n=10):
        """PageRank 기반 인기 영화 추천"""
        with self.driver.session() as session:
            query = """
            MATCH (m:Movie)
            WHERE m.pagerank IS NOT NULL
            RETURN m.title AS title, 
                   m.pagerank AS score,
                   m.voteAverage AS rating,
                   m.voteCount AS voteCount
            ORDER BY m.pagerank DESC
            LIMIT $top_n
            """
            result = session.run(query, top_n=top_n)
            
            recommendations = []
            for record in result:
                recommendations.append({
                    'title': record['title'],
                    'score': record['score'],
                    'rating': record['rating'],
                    'voteCount': record['voteCount']
                })
            return recommendations
    
    def recommend_by_similarity(self, movie_title, top_n=5):
        """유사 영화 기반 추천"""
        with self.driver.session() as session:
            query = """
            MATCH (m1:Movie {title: $title})-[s:SIMILAR]-(m2:Movie)
            WHERE NOT EXISTS {
                MATCH (m1)<-[:RATED]-(:User)-[:RATED]->(m2)
            }
            RETURN m2.title AS title,
                   s.similarity AS score,
                   m2.voteAverage AS rating
            ORDER BY s.similarity DESC
            LIMIT $top_n
            """
            result = session.run(query, title=movie_title, top_n=top_n)
            
            recommendations = []
            for record in result:
                recommendations.append({
                    'title': record['title'],
                    'score': record['score'],
                    'rating': record['rating']
                })
            return recommendations
    
    def recommend_by_community(self, user_id, top_n=10):
        """커뮤니티 기반 추천"""
        with self.driver.session() as session:
            # 사용자가 본 영화들의 커뮤니티 찾기
            query = """
            MATCH (u:User {userId: $user_id})-[:RATED]->(m:Movie)
            WHERE m.community IS NOT NULL
            WITH m.community AS community, count(*) AS count
            ORDER BY count DESC
            LIMIT 1
            WITH community
            MATCH (m:Movie {community: community})
            WHERE NOT EXISTS {
                MATCH (u:User {userId: $user_id})-[:RATED]->(m)
            }
            RETURN m.title AS title,
                   m.voteAverage AS rating,
                   m.pagerank AS popularity
            ORDER BY m.pagerank DESC
            LIMIT $top_n
            """
            result = session.run(query, user_id=user_id, top_n=top_n)
            
            recommendations = []
            for record in result:
                recommendations.append({
                    'title': record['title'],
                    'rating': record['rating'],
                    'popularity': record['popularity']
                })
            return recommendations

# 추천 시스템 객체 생성
recommender = MovieRecommender(driver, gds)
print("추천 시스템 준비 완료!")


In [ ]:
# ============================================
# 추천 시스템 테스트
# ============================================

print("=" * 60)
print("1. 인기 영화 추천 (PageRank 기반)")
print("=" * 60)
popular_movies = recommender.recommend_by_popularity(top_n=10)
for i, movie in enumerate(popular_movies, 1):
    print(f"{i:2d}. {movie['title']}")
    print(f"    PageRank: {movie['score']:.6f}, 평점: {movie['rating']:.1f}, 평가수: {movie['voteCount']}")

print("\n" + "=" * 60)
print("2. 유사 영화 추천 (Node Similarity 기반)")
print("=" * 60)
similar_movies = recommender.recommend_by_similarity("Toy Story", top_n=5)
if similar_movies:
    print("'Toy Story'와 유사한 영화:")
    for i, movie in enumerate(similar_movies, 1):
        print(f"{i:2d}. {movie['title']}")
        print(f"    유사도: {movie['score']:.4f}, 평점: {movie['rating']:.1f}")
else:
    print("유사 영화를 찾을 수 없습니다.")

print("\n" + "=" * 60)
print("3. 커뮤니티 기반 추천")
print("=" * 60)
# 사용자 ID 확인
with driver.session() as session:
    query = "MATCH (u:User) RETURN u.userId AS userId LIMIT 1"
    result = session.run(query)
    record = result.single()
    if record:
        user_id = record['userId']
        community_movies = recommender.recommend_by_community(user_id, top_n=5)
        if community_movies:
            print(f"사용자 {user_id}를 위한 커뮤니티 기반 추천:")
            for i, movie in enumerate(community_movies, 1):
                print(f"{i:2d}. {movie['title']}")
                print(f"    평점: {movie['rating']:.1f}, 인기도: {movie['popularity']:.6f}")
        else:
            print("추천할 영화를 찾을 수 없습니다.")


## 11. 그래프 프로젝션 정리

작업이 끝나면 메모리에서 그래프 프로젝션을 삭제하는 것이 좋습니다.


In [ ]:
# ============================================
# 그래프 프로젝션 삭제
# ============================================
# 메모리 절약을 위해 사용하지 않는 그래프 프로젝션을 삭제합니다.

# 현재 프로젝션된 그래프 목록 확인
graphs = gds.graph.list()
print("현재 프로젝션된 그래프:")
for graph_name in graphs:
    print(f"  - {graph_name}")


In [ ]:
# 그래프 삭제
gds.graph.drop("movies-users-graph")
gds.graph.drop("movies-genres-graph")
print("그래프 프로젝션 삭제 완료!")


In [ ]:
# ============================================
# 연결 종료
# ============================================
driver.close()
print("Neo4j 연결 종료 완료!")


### 참고 자료
- [Neo4j GDS 공식 문서](https://neo4j.com/docs/graph-data-science/)
- [GDS 알고리즘 가이드](https://neo4j.com/docs/graph-data-science/current/algorithms/)
- [Python graphdatascience 라이브러리](https://neo4j.com/docs/graph-data-science-client/python/)
